# MBC 뉴스 본문 수집 (BS4 Colab용) — 사이트 직접 / 기간 모드

URL 수집 노트북에서 만든 `링크_{press}_*.json` 파일을 읽어 MBC 사이트(`imnews.imbc.com`) article 페이지를 BS4로 파싱한다. 통신3사/LPOD/SBS/KBS 트랙과 동일하게 **기간 단위 통합 CSV** 1개를 만든다 (입력 JSON 1개 → 출력 CSV 1개). MBC는 og:* / article:* 메타 태그가 잘 갖춰져 있어 SBS와 비슷하게 메타에서 대부분 필드를 얻는다.

- 입력: `data/링크_{press}_{YYMMDD}_{YYMMDD}.json` (기간 통합)
- 출력: `data/본문_bs4_{press}_{YYMMDD}_{YYMMDD}.csv`
- 보조 출력: 중간 재개용 체크포인트 JSON, 재실패 URL JSON
- 특징: 기간 단위 통합 수집, 중간 재개, 오류 URL 1회 재시도, 완료 파일 건너뛰기
- 셀렉터: title=og:title, body=div.news_txt (=`div[itemprop="articleBody"]`), pubdate=article:published_time, category=article:section

In [1]:
# Colab 환경 세팅 — requests, BeautifulSoup 설치
# MBC article 페이지는 정적 HTML이라 Selenium 불필요, requests + BS4로 충분
# !pip install -q requests beautifulsoup4 pandas

In [2]:
# Google Drive 마운트 — 중간에 끊겨도 데이터 보존
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
# Drive 안의 프로젝트 폴더로 이동
# 통신3사 트리와 분리하기 위해 KBS/SBS/MBC 등 언론사 직접 수집은 news/ 하위에 보관
import os
PROJECT_DIR = '/content/drive/MyDrive/Text-data-Analysis_26-Spring/news'
os.chdir(PROJECT_DIR)
print(f'현재 작업 폴더: {os.getcwd()}')

현재 작업 폴더: /content/drive/MyDrive/Text-data-Analysis_26-Spring/news


In [4]:
import json
import os
import random
import re
import time
from datetime import datetime, timedelta
from pathlib import Path

import pandas as pd
import requests
from bs4 import BeautifulSoup

# 로컬/Colab 비교를 위해 User-Agent 고정
USER_AGENT = 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/147.0.0.0 Safari/537.36'

# 담당 언론사와 수집 기간 지정 — 날짜 형식: 'YYYY.MM.DD'
# URL 수집 노트북과 동일한 press_ranges를 그대로 사용
# press 하나당 start_date ~ end_date 안의 일자를 한 통합 CSV로 묶어 저장
# _direct 접미사로 네이버 경유본 본문(`본문_bs4_MBC_*.csv`)과 출력 파일 분리
press_ranges = [
    {'press': 'MBC_direct', 'start_date': '2026.05.05', 'end_date': '2026.05.11'},
]


# 기간 단위 작업 목록 생성 — press_ranges 한 항목당 jobs 1개
# URL 수집 노트북과 같은 시그니처(press/start_date/end_date)를 본문 수집에 그대로 전달
def build_period_jobs(press_ranges):
    jobs = []
    for item in press_ranges:
        # 'YYYY.MM.DD' 문자열을 datetime으로 파싱 — 기간 유효성 검사용
        start = datetime.strptime(item['start_date'], '%Y.%m.%d')
        end = datetime.strptime(item['end_date'], '%Y.%m.%d')
        # 시작 일자가 끝 일자보다 늦으면 작업 범위가 잘못된 것이므로 즉시 중단
        if start > end:
            raise ValueError(f"시작 일자가 끝 일자보다 늦습니다: {item}")
        jobs.append({
            'press': item['press'],
            'start_date': item['start_date'],
            'end_date': item['end_date'],
        })
    return jobs


# 변수 정의 (날짜 형식: 'YYYY.MM.DD')
jobs = build_period_jobs(press_ranges)

print(f'총 작업 수: {len(jobs)}')
for job in jobs:
    print(job)

# 셀 3을 건너뛰고 실행해도 기본 프로젝트 경로를 사용할 수 있게 보완
try:
    PROJECT_DIR
except NameError:
    PROJECT_DIR = '/content/drive/MyDrive/Text-data-Analysis_26-Spring/news'

# 저장할 폴더 지정 — 링크 파일, 체크포인트, 본문 CSV, 실패 목록이 모두 이 폴더에 저장
SAVE_DIR = Path(PROJECT_DIR) / 'notebook' / 'crawling' / 'data'
SAVE_DIR.mkdir(parents=True, exist_ok=True)
print(f'저장 위치: {SAVE_DIR}')

# requests 세션 — User-Agent/Accept-Language를 반복 요청에 일관 적용
session = requests.Session()
session.headers.update({
    'User-Agent': USER_AGENT,
    'Accept-Language': 'ko-KR,ko;q=0.9,en-US;q=0.8,en;q=0.7',
})
print(f'User-Agent: {USER_AGENT}')

총 작업 수: 1
{'press': 'MBC_direct', 'start_date': '2026.05.05', 'end_date': '2026.05.11'}
저장 위치: /content/drive/MyDrive/Text-data-Analysis_26-Spring/news/notebook/crawling/data
User-Agent: Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/147.0.0.0 Safari/537.36


In [5]:
# 봇 탐지 방지를 위해 기사/job 사이에 랜덤한 시간을 기다림
ARTICLE_PAUSE_RANGE_SEC = (0.4, 1.2)
JOB_PAUSE_RANGE_SEC = (8, 20)

# 중간에 끊겨도 이어서 수집할 수 있도록 일정 건수마다 체크포인트 저장
CHECKPOINT_INTERVAL = 200
# requests 응답 대기 상한 — MBC 페이지가 늦게 뜨면 끊고 err_idx에 기록
REQUEST_TIMEOUT_SEC = 15
SKIP_COMPLETED = True


# 랜덤 대기 후 로그 출력
def polite_sleep(label, pause_range):
    pause_sec = random.uniform(*pause_range)
    print(f"{label} {pause_sec:.1f}초 대기")
    time.sleep(pause_sec)


# 파일명에 사용할 YYMMDD_YYMMDD 형식 기간 문자열 생성 (통신3사 트랙과 동일 시그니처)
# 예: 2026.05.01 ~ 2026.05.07 -> '260501_260507'
def make_period_suffix(start_date, end_date):
    return f"{start_date.replace('.', '')[2:]}_{end_date.replace('.', '')[2:]}"


# 지정한 meta property의 content 반환 (없으면 빈 문자열)
# MBC는 SBS와 비슷하게 og:* / article:* 메타가 잘 갖춰져 있어 대부분 필드를 여기서 추출
def meta_content(soup, prop):
    el = soup.select_one(f'meta[property="{prop}"]')
    return el.get('content', '') if el else ''


# 본문 안의 줄바꿈/연속 공백을 하나의 공백으로 정리 (CSV/분석 단계 일관성 위해)
def normalize_body_text(text):
    return re.sub(r'\s+', ' ', text).strip()


# 기사 한 건에서 title/body/pubdate/category 추출 (MBC 셀렉터 사용)
# title/body/pubdate 중 하나라도 비면 호출부에서 err_idx로 기록할 수 있도록 ValueError 발생
def extract_article_bs4(link, session=session):
    # MBC 사이트 기사 페이지 HTML 요청 — 정적 HTML이라 requests로 충분
    response = session.get(link, timeout=REQUEST_TIMEOUT_SEC)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, 'html.parser')

    # 제목 추출하기 — og:title 메타 태그 (HTML entity는 BS4가 자동 디코딩)
    title = meta_content(soup, 'og:title')

    # 본문 추출하기 — div.news_txt (= div[itemprop="articleBody"], schema.org microdata)
    body_el = soup.select_one('div.news_txt')
    body = normalize_body_text(body_el.get_text(' ', strip=True)) if body_el else ''

    # 날짜 추출하기 — article:published_time (ISO 8601, 시간대 +09:00 포함)
    # 예: '2026-05-05T22:53:44+09:00' — pd.to_datetime이 자동 파싱
    pubdate = meta_content(soup, 'article:published_time')

    # 카테고리: article:section ('정치', '경제' 등 한글 라벨)
    category = meta_content(soup, 'article:section')

    # 제목/본문/날짜 중 하나라도 없으면 실패 — 호출부에서 err_idx로 기록
    if not title or not body or not pubdate:
        raise ValueError('title/body/pubdate 중 일부를 추출하지 못함')

    return {
        'link': link,
        'pubdate': pubdate,
        'category': category,
        'title': title,
        'body': body,
    }


# 한 언론사의 기간 통합 링크 JSON을 읽어 본문을 수집하고 통합 CSV로 저장
# 체크포인트 기반 재개 + 오류 자동 1회 재시도 + 재실패 URL JSON 저장
def collect_bodies_bs4(press, start_date, end_date, save_dir=SAVE_DIR):
    # 파일명 키로 쓸 기간 접미사 (예: 260501_260507)
    period = make_period_suffix(start_date, end_date)
    # 입력: URL 수집 단계가 만들어 둔 기간 통합 MBC 링크 JSON
    links_path = save_dir / f"링크_{press}_{period}.json"
    # 체크포인트: 중간 결과(all_results) + 오류 인덱스 + 다음 인덱스를 보관
    checkpoint_path = save_dir / f"체크포인트_본문_bs4_{press}_{period}.json"
    # 최종 출력: 기간 통합 본문 CSV (네이버 경유본과 분리하기 위해 press에 _direct 접미사가 붙음)
    csv_save_path = save_dir / f"본문_bs4_{press}_{period}.csv"

    # 최종 CSV가 이미 있으면 같은 기간은 건너뜀 (재실행 시 idempotent)
    if SKIP_COMPLETED and csv_save_path.exists():
        print()
        print(f"=== {press} / {start_date} ~ {end_date} 이미 완료됨, 건너뜀 ===")
        print(f"기존 파일: {csv_save_path}")
        return csv_save_path

    # 링크 파일 불러오기
    if links_path.exists():
        with links_path.open('r', encoding='utf-8') as f:
            mbc_news_links = json.load(f)
        # 기사 한 건당 대략 (랜덤 대기 평균 + 0.7s) 잡은 거친 예상치 — 로그 확인용
        avg_pause_sec = sum(ARTICLE_PAUSE_RANGE_SEC) / 2
        est_sec_per_article = avg_pause_sec + 0.7
        est_min = len(mbc_news_links) * est_sec_per_article / 60
        print()
        print(f"=== {press} / {start_date} ~ {end_date} BS4 본문 수집 시작 ===")
        print(f'링크 {len(mbc_news_links)}개 불러옴: {links_path}')
        print(f'예상 소요 시간: 약 {est_min:.0f}분')
    else:
        raise FileNotFoundError(f'링크 파일 없음 — MBC_직접_url_수집_colab.ipynb를 먼저 실행하세요\n경로: {links_path}')

    # 이전에 중단된 작업이 있으면 이어받기 — next_i 인덱스 다음부터 시작
    if checkpoint_path.exists():
        with checkpoint_path.open('r', encoding='utf-8') as f:
            checkpoint = json.load(f)
        # JSON은 dict 키를 문자열로 저장하므로 int로 다시 변환
        all_results = {int(k): v for k, v in checkpoint.get('all_results', {}).items()}
        err_idx = checkpoint.get('err_idx', [])
        i = checkpoint.get('next_i', 0)
        print(f'체크포인트 발견 — {i}번째부터 이어서 시작 (이미 수집: {len(all_results)}건)')
    else:
        all_results = dict()
        i = 0
        err_idx = []
        print('새로 시작')

    # 본문 수집 메인 루프 — i 인덱스를 함께 들고 다녀 체크포인트와 동기화
    for link in mbc_news_links[i:]:
        try:
            all_results[i] = extract_article_bs4(link)
            # 진행 상황 확인용 코드
            print(f'[{i+1} / {len(mbc_news_links)}] \t {(i+1)/len(mbc_news_links)*100:.2f}% \t error: {len(err_idx)}')
            i += 1
            # 중간저장 — N건마다 체크포인트 갱신해 중간 중단에도 진행 보존
            if i % CHECKPOINT_INTERVAL == 0:
                with checkpoint_path.open('w', encoding='utf-8') as f:
                    json.dump({'all_results': all_results, 'err_idx': err_idx, 'next_i': i}, f, ensure_ascii=False, indent=2)
                print(f'체크포인트 저장 — {i}건 완료')
            polite_sleep('다음 기사 전', ARTICLE_PAUSE_RANGE_SEC)
        except Exception as exc:
            # 오류 인덱스를 err_idx에 누적해두고 즉시 체크포인트도 갱신
            print(f'오류 발생 — index {i}: {exc!r}')
            err_idx.append(i)
            i += 1
            with checkpoint_path.open('w', encoding='utf-8') as f:
                json.dump({'all_results': all_results, 'err_idx': err_idx, 'next_i': i}, f, ensure_ascii=False, indent=2)

    # 1차 오류 자동 재시도 — 페이지가 늦게 떴거나 일시적 네트워크 문제였던 경우를 한 번 더 시도
    if err_idx:
        print()
        print(f'오류 {len(err_idx)}건 재시도 시작...')
        re_err_idx = []
        for retry_i in err_idx:
            try:
                link = mbc_news_links[retry_i]
                all_results[retry_i] = extract_article_bs4(link)
                print(f'재시도 성공 — index {retry_i}')
                polite_sleep('다음 재시도 전', ARTICLE_PAUSE_RANGE_SEC)
            except Exception as exc:
                print(f'재시도 실패 — index {retry_i}: {exc!r}')
                re_err_idx.append(retry_i)
        # 재시도 후에도 실패한 인덱스만 남김 — 본문_bs4_재실패_*.json으로 저장 대상
        err_idx = re_err_idx
        print(f'재시도 완료 — 재실패: {len(err_idx)}건')

    # 수집한 정보들을 dataframe으로 변환
    df = pd.DataFrame(all_results).T
    if df.empty:
        raise ValueError('수집된 본문 데이터가 없습니다.')

    # 수집한 기사들 중 중복인 경우 이를 제거
    df_no_duplicates = df.drop_duplicates().reset_index(drop=True)
    # 오래된 순부터 수집했으나 혹시 모를 상황을 방지하기 위해 pubdate를 datetime으로 변환 후 정렬
    df_no_duplicates['pubdate'] = pd.to_datetime(df_no_duplicates['pubdate'], errors='coerce')
    df_sorted = df_no_duplicates.sort_values(by='pubdate')

    # 수집한 정보들을 csv로 저장 (Google Drive에 저장)
    df_sorted.to_csv(csv_save_path, index=False, encoding='utf-8-sig')
    print(f'저장 완료: {csv_save_path}')

    if err_idx:
        # 재시도 후에도 실패한 URL은 별 JSON으로 떨어뜨려 Selenium 재시도 노트북에서 다시 시도 가능
        failed_path = save_dir / f"본문_bs4_재실패_{press}_{period}.json"
        with failed_path.open('w', encoding='utf-8') as f:
            json.dump({'err_idx': err_idx, 'links': [mbc_news_links[x] for x in err_idx]}, f, ensure_ascii=False, indent=2)
        print(f'재실패 목록 저장: {failed_path}')
    elif checkpoint_path.exists():
        # 재실패가 없을 때만 체크포인트 삭제 — 실패가 남아있으면 디버깅용으로 보존
        checkpoint_path.unlink()

    print(f'본문 수집 완료 — 총 {len(df_sorted)}건 / 오류 {len(err_idx)}건')
    return csv_save_path


# 생성된 jobs를 순서대로 실행
# 한 작업이 실패해도 실패 목록에 기록하고 다음 작업으로 넘어감
results = []
failures = []
for index, job in enumerate(jobs, start=1):
    print()
    print(f"[{index}/{len(jobs)}] 작업 실행: {job}")
    try:
        # job 딕셔너리의 press/start_date/end_date를 collect_bodies_bs4 인자로 전달
        results.append(collect_bodies_bs4(**job))
    except Exception as exc:
        # 한 언론사에서 오류가 나도 전체 작업이 멈추지 않도록 실패 정보만 저장
        failures.append({'job': job, 'error': repr(exc)})
        print(f"작업 실패, 다음 작업으로 넘어감: {exc!r}")
    finally:
        if index < len(jobs):
            # 다음 언론사 job으로 넘어가기 전 대기
            polite_sleep('다음 작업 전', JOB_PAUSE_RANGE_SEC)

# 실패한 작업이 있으면 나중에 다시 돌릴 수 있게 파일로 저장
if failures:
    failures_path = SAVE_DIR / '본문_bs4_수집실패목록_MBC_direct.json'
    with failures_path.open('w', encoding='utf-8') as f:
        json.dump(failures, f, ensure_ascii=False, indent=2)
    print()
    print(f"실패 작업 {len(failures)}개 저장: {failures_path}")

print()
print('전체 작업 완료')
print(f'성공/건너뜀: {len(results)}개, 실패: {len(failures)}개')
for result_path in results:
    print(result_path)


[1/1] 작업 실행: {'press': 'MBC_direct', 'start_date': '2026.05.05', 'end_date': '2026.05.11'}

=== MBC_direct / 2026.05.05 ~ 2026.05.11 BS4 본문 수집 시작 ===
링크 242개 불러옴: /content/drive/MyDrive/Text-data-Analysis_26-Spring/news/notebook/crawling/data/링크_MBC_direct_260505_260511.json
예상 소요 시간: 약 6분
새로 시작
[1 / 242] 	 0.41% 	 error: 0
다음 기사 전 0.6초 대기
[2 / 242] 	 0.83% 	 error: 0
다음 기사 전 0.8초 대기
[3 / 242] 	 1.24% 	 error: 0
다음 기사 전 1.1초 대기
[4 / 242] 	 1.65% 	 error: 0
다음 기사 전 0.8초 대기
[5 / 242] 	 2.07% 	 error: 0
다음 기사 전 0.8초 대기
[6 / 242] 	 2.48% 	 error: 0
다음 기사 전 0.7초 대기
[7 / 242] 	 2.89% 	 error: 0
다음 기사 전 0.5초 대기
[8 / 242] 	 3.31% 	 error: 0
다음 기사 전 1.1초 대기
[9 / 242] 	 3.72% 	 error: 0
다음 기사 전 0.5초 대기
[10 / 242] 	 4.13% 	 error: 0
다음 기사 전 0.4초 대기
[11 / 242] 	 4.55% 	 error: 0
다음 기사 전 0.9초 대기
[12 / 242] 	 4.96% 	 error: 0
다음 기사 전 0.8초 대기
[13 / 242] 	 5.37% 	 error: 0
다음 기사 전 0.6초 대기
[14 / 242] 	 5.79% 	 error: 0
다음 기사 전 0.5초 대기
[15 / 242] 	 6.20% 	 error: 0
다음 기사 전 0.7초 대기
[16 / 242] 	 6.61% 	 